# Modelos interpretables para el valor por AGEB

Complemento de `precio_estimado.ipynb` (sesión 23c de Mapo). Ahí se eligió la forma del precio con un bosque aleatorio; aquí se prueba si se puede llegar a lo mismo con modelos que se puedan leer, siguiendo los temas de Análisis Estadístico Multivariado, y se agregan tres interpretables de fuera del temario, una red neuronal chica y dos cajas negras de referencia.

- **Objetivo:** `log(medv)`, mediana del valor catastral de suelo de zona en la AGEB ($/m²), con el municipio como factor (absorbe que las tablas son de años distintos).
- **Validación:** cada AGEB se predice con un modelo que no vio su bloque de 3 km. Los hiperparámetros también se eligen por bloques.
- **Dos medidas:** RMSE en log (error relativo típico) y Spearman dentro de cada municipio (si el orden de las AGEB sale bien).

In [1]:
%load_ext autoreload
%autoreload 2

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from precio import conformal, datos, graficas, interpretables, modelos

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)

RAIZ = Path("../..")
DATASET = RAIZ / "mapo_core/data/vivienda_gdl_v7.zip"
ALFA = 0.2  # rango con 80 % de cobertura nominal
RASGOS = datos.RASGOS

## $\color{#dda}{\text{1. Datos y correlaciones}}$

Las mismas 892 AGEB con catastro del estudio principal y sus 44 rasgos. Antes de modelar: qué rasgos se mueven con el valor.

In [2]:
v = datos.cargar(DATASET)
v["bloque"] = datos.bloques(v)
base = v.dropna(subset=["y"]).reset_index(drop=True)
X, y, g = base[RASGOS + ["municipio"]], base["y"], base["bloque"]
base.groupby("municipio").agg(agebs=("cvegeo", "size"), mediana=("medv", "median"))

,agebs,mediana
municipio,,
Guadalajara,415,3080.0
Ixtlahuacán de los Membrillos,23,2140.0
Tonalá,152,2172.5
Zapopan,302,3750.0


In [3]:
graficas.correlaciones(base, RASGOS)

In [4]:
z = interpretables.estandarizar(base, RASGOS)
fuertes = base[RASGOS].corrwith(base["y"], method="spearman").abs().nlargest(12).index
graficas.mapa_calor(z[fuertes].corr(), "Correlación entre los 12 rasgos más ligados al valor", "r")

## $\color{#dda}{\text{2. Distribución del objetivo}}$

El valor en pesos tiene cola larga a la derecha; en logaritmo queda cerca de simétrico. Por eso se modela `log(valor)` y los coeficientes se leen como porcentajes.

In [5]:
pd.DataFrame({"sesgo": [base["medv"].skew(), base["y"].skew()],
              "curtosis": [base["medv"].kurt(), base["y"].kurt()]},
             index=["valor ($/m²)", "log(valor)"])

,sesgo,curtosis
valor ($/m²),2.726,13.911
log(valor),-1.251,3.095


In [6]:
graficas.distribucion(base)

## $\color{#dda}{\text{3. Regresión lineal por máxima verosimilitud}}$

OLS con los 44 rasgos estandarizados y el municipio como factor. Si el error es normal, OLS es el estimador de máxima verosimilitud. Los errores estándar van agrupados por bloque de 3 km: dos AGEB vecinas no son dos observaciones independientes, y tratarlas así daría intervalos demasiado angostos.

In [7]:
res = interpretables.ols(base, RASGOS)
print(f"R² = {res.rsquared:.3f}, log-verosimilitud = {res.llf:.1f}, AIC = {res.aic:.1f}, BIC = {res.bic:.1f}")
coef = interpretables.coeficientes(res, RASGOS)
coef.head(15)

R² = 0.456, log-verosimilitud = -778.9, AIC = 1653.8, BIC = 1883.9


,coef,bajo,alto,p,cambio_pct
graproes,0.152,0.032,0.273,1.289e-02,16.462
pro_ocup_c,-0.135,-0.264,-0.005,4.174e-02,-12.590
pct_viv_internet,0.116,-0.076,0.308,2.378e-01,12.280
age_1975,0.101,-0.157,0.360,4.421e-01,10.667
dis_km,-0.099,-0.266,0.067,2.427e-01,-9.449
pct_viv_uso_temporal,-0.094,-0.181,-0.007,3.350e-02,-8.973
pendiente_grados,-0.093,-0.153,-0.034,2.101e-03,-8.926
parques_ha_1km,0.089,0.036,0.142,9.693e-04,9.323
razon_dependencia,0.081,-0.006,0.169,6.913e-02,8.485
negocios_1km,-0.081,-0.292,0.130,4.518e-01,-7.769


In [8]:
graficas.coeficientes(coef.head(20), "OLS completa: efecto de cada rasgo (los 20 más grandes)")

In [9]:
graficas.qq(res.resid, "Residuos de la OLS contra la normal")

Con 44 rasgos correlacionados muchos signos son inestables. El VIF mide cuánto se infla la varianza de cada coeficiente por la colinealidad (arriba de 10 es grave).

In [10]:
interpretables.vif(base, RASGOS).head(12)

age_1975                  32.520
age                       32.078
viv_hab                   31.969
viv_ha                    30.839
negocios_1km              21.035
dis_km                    19.365
salud_1km                 17.753
area_ha                   16.083
restaurantes_cafes_1km    15.162
nox                       13.407
graproes                  11.954
pct_viv_3ymas_cuartos     11.133
Name: vif, dtype: float64

## $\color{#dda}{\text{4. Selección de variables}}$

Dos caminos. **Por pasos:** se agrega el rasgo que más baja el BIC hasta que ninguno lo baje. **Lasso:** penaliza el tamaño de los coeficientes y deja en cero los que no aportan; la penalización se elige por validación cruzada.

In [11]:
pasos = interpretables.Pasos(RASGOS).fit(X, y)
elegidos = pasos.elegidos_
print(len(elegidos), "rasgos por pasos:", elegidos)
res_pasos = interpretables.ols(base, elegidos)
coef_pasos = interpretables.coeficientes(res_pasos, elegidos)
coef_pasos

11 rasgos por pasos: ['pct_viv_internet', 'pct_60ymas', 'diversidad_giros_1km', 'pct_viv_2ymas_dorm', 'parques_ha_1km', 'pendiente_grados', 'area_ha', 'pro_ocup_c', 'rad_km', 'pct_viv_uso_temporal', 'graproes']


,coef,bajo,alto,p,cambio_pct
pro_ocup_c,-0.138,-0.223,-5.338e-02,1.413e-03,-12.917
graproes,0.118,0.034,2.016e-01,5.720e-03,12.518
pendiente_grados,-0.110,-0.157,-6.315e-02,4.058e-06,-10.406
pct_viv_internet,0.107,-0.021,2.342e-01,1.017e-01,11.248
parques_ha_1km,0.096,0.049,1.440e-01,6.807e-05,10.130
area_ha,0.076,0.021,1.311e-01,6.761e-03,7.901
pct_viv_uso_temporal,-0.073,-0.146,5.986e-04,5.190e-02,-7.030
pct_60ymas,0.072,-0.003,1.469e-01,5.927e-02,7.471
pct_viv_2ymas_dorm,-0.069,-0.130,-8.703e-03,2.494e-02,-6.679
rad_km,-0.068,-0.114,-2.260e-02,3.370e-03,-6.590


In [12]:
graficas.coeficientes(coef_pasos, "OLS con los rasgos elegidos por BIC")

In [13]:
interpretables.vif(base, elegidos)

pct_viv_internet        5.847
graproes                3.787
pro_ocup_c              3.577
pct_60ymas              2.126
diversidad_giros_1km    1.695
pct_viv_2ymas_dorm      1.636
parques_ha_1km          1.437
pct_viv_uso_temporal    1.243
pendiente_grados        1.211
rad_km                  1.098
area_ha                 1.094
Name: vif, dtype: float64

In [14]:
cat = interpretables.catalogo(RASGOS)
lasso = modelos.ajustar(cat["Lasso"][1], X, y, g)
coef_lasso = interpretables.coef_lasso(lasso)
print(f"Lasso deja {len(coef_lasso)} de {len(RASGOS)} rasgos (más indicadores de dato faltante)")
coef_lasso.head(20)

Lasso deja 20 de 44 rasgos (más indicadores de dato faltante)


,coef,cambio_pct
pro_ocup_c,-1.264e-01,-11.873
graproes,1.001e-01,10.527
pct_viv_internet,7.128e-02,7.388
parques_ha_1km,7.128e-02,7.388
pendiente_grados,-6.639e-02,-6.423
area_ha,6.340e-02,6.545
pct_viv_uso_temporal,-5.501e-02,-5.352
diversidad_giros_1km,5.327e-02,5.471
supermercados_1km,4.300e-02,4.394
pct_60ymas,3.047e-02,3.094


## $\color{#dda}{\text{5. ANOVA y ANCOVA por municipio}}$

¿Cuánto del valor explica solo el municipio? ANOVA de una vía. Después, ANCOVA: el municipio con los rasgos elegidos como covariables. Al final, si la pendiente del rasgo más fuerte cambia entre municipios (el supuesto de pendientes paralelas de la ANCOVA).

In [15]:
anova = interpretables.anova_municipio(base, elegidos)
anova["una_via"]

,sum_sq,df,F,PR(>F),eta2
C(municipio),62.364,3.0,37.835,5.163e-23,0.113
Residual,487.904,888.0,NaN,NaN,NaN


In [16]:
anova["ancova"]

,sum_sq,df,F,PR(>F)
C(municipio),1.385,3.0,1.299,2.734e-01
pct_viv_internet,1.680,1.0,4.729,2.992e-02
pct_60ymas,1.681,1.0,4.732,2.987e-02
diversidad_giros_1km,1.304,1.0,3.671,5.569e-02
pct_viv_2ymas_dorm,2.533,1.0,7.132,7.711e-03
parques_ha_1km,5.652,1.0,15.914,7.181e-05
pendiente_grados,8.714,1.0,24.535,8.755e-07
area_ha,4.537,1.0,12.773,3.708e-04
pro_ocup_c,4.727,1.0,13.309,2.798e-04
rad_km,3.469,1.0,9.767,1.835e-03


In [17]:
anova["pendientes"]

,df_resid,ssr,df_diff,ss_diff,F,Pr(>F)
0,877.0,311.486,0.0,NaN,NaN,NaN
1,874.0,308.608,3.0,2.878,2.717,0.044


## $\color{#dda}{\text{6. Componentes principales}}$

Los 44 rasgos comparten mucha información. PCA los resume en ejes sin correlación; la regresión sobre componentes (PCR) usa los primeros. Las cargas dicen qué significa cada eje.

In [18]:
var, cargas = interpretables.pca(base, RASGOS, n=3)
graficas.varianza(var)

In [19]:
top = cargas.abs().max(axis=1).nlargest(20).index
graficas.mapa_calor(cargas.loc[top], "Cargas de los tres primeros componentes", "carga")

## $\color{#dda}{\text{7. Tipos de zona con k-medias}}$

Agrupar AGEB parecidas en tipos de zona y ver si el tipo, junto con el municipio, basta para explicar el valor (una ANOVA de dos factores).

In [20]:
etiqueta, perfil = interpretables.tipos(base, RASGOS, k=6)
base.assign(tipo=etiqueta).groupby("tipo").agg(agebs=("cvegeo", "size"), mediana=("medv", "median"))

,agebs,mediana
tipo,,
0,142,2090.0
1,13,950.0
2,186,5462.5
3,79,1700.0
4,364,2960.0
5,108,3750.0


In [21]:
destaca = perfil.abs().max(axis=1).nlargest(18).index
graficas.mapa_calor(perfil.loc[destaca], "Perfil de cada tipo (media estandarizada)", "z")

## $\color{#dda}{\text{8. Regresión logística: qué hace cara a una AGEB}}$

Otra pregunta, más fácil de contar: la probabilidad de que una AGEB valga más que la mediana de su municipio. Razón de momios por una desviación estándar de cada rasgo elegido; arriba de 1, sube la probabilidad.

In [22]:
momios, auc = interpretables.logistica(base, elegidos)
print(f"AUC fuera de muestra por bloques: {auc:.3f}")
momios

AUC fuera de muestra por bloques: 0.798


,razon_momios,bajo,alto,p
pro_ocup_c,0.595,0.398,0.889,0.011
rad_km,0.663,0.511,0.859,0.002
pct_viv_2ymas_dorm,0.695,0.542,0.892,0.004
pct_viv_uso_temporal,0.733,0.592,0.908,0.004
pendiente_grados,0.786,0.633,0.976,0.029
pct_viv_internet,0.939,0.513,1.718,0.837
area_ha,0.990,0.782,1.253,0.931
parques_ha_1km,1.078,0.852,1.364,0.533
pct_60ymas,1.234,0.877,1.736,0.227
diversidad_giros_1km,1.316,0.998,1.737,0.052


## $\color{#dda}{\text{9. Comparación fuera de muestra}}$

Todos con la misma validación por bloques. Familias: del temario de la clase, interpretables de fuera del temario, la red y las dos cajas negras de referencia (SVR y bosque, las del estudio principal).

In [23]:
predicciones, tablas = {}, []
for nombre, (familia, modelo) in cat.items():
    predicciones[nombre] = modelos.fuera_de_muestra(modelo, X, y, g)
    tablas.append(modelos.metricas(y, predicciones[nombre], base["municipio"]).assign(modelo=nombre))
comparacion = pd.concat(tablas)
familias = {n: f for n, (f, _) in cat.items()}
comparacion.pivot(index="modelo", columns="municipio", values="rmse_log").sort_values("Todos")

municipio,Guadalajara,Ixtlahuacán de los Membrillos,Todos,Tonalá,Zapopan
modelo,,,,,
Lasso,0.226,0.892,0.619,0.730,0.857
Bosque,0.196,0.862,0.622,0.733,0.873
Aditivo (splines),0.216,0.885,0.629,0.727,0.883
Pasos BIC,0.257,1.004,0.634,0.719,0.872
SVR,0.210,0.956,0.636,0.754,0.883
PCR,0.228,1.051,0.643,0.744,0.888
Tipos k-medias,0.270,0.971,0.653,0.786,0.882
Mixto,0.270,0.949,0.658,0.766,0.903
OLS (ANCOVA),0.271,0.985,0.663,0.769,0.908


In [24]:
comparacion.pivot(index="modelo", columns="municipio", values="spearman").sort_values("Todos", ascending=False)

municipio,Guadalajara,Ixtlahuacán de los Membrillos,Todos,Tonalá,Zapopan
modelo,,,,,
Lasso,0.810,0.115,0.738,0.496,0.669
SVR,0.799,0.038,0.735,0.460,0.639
Pasos BIC,0.788,-0.131,0.722,0.457,0.643
Bosque,0.823,0.119,0.720,0.452,0.628
Aditivo (splines),0.812,0.036,0.717,0.478,0.625
PCR,0.816,-0.050,0.716,0.489,0.625
Mixto,0.781,0.092,0.698,0.389,0.615
OLS (ANCOVA),0.783,0.088,0.695,0.397,0.609
Red (32-16),0.707,0.537,0.654,0.346,0.603


In [25]:
graficas.modelos_juntos(comparacion.query("municipio == 'Todos'"), familias)

In [26]:
arbol = modelos.ajustar(cat["Árbol"][1], X, y, g)
print(interpretables.reglas_arbol(arbol, RASGOS))

|--- pro_ocup_c <= 0.94
|   |--- graproes <= 13.47
|   |   |--- dis_km <= 2.58
|   |   |   |--- pct_nacidos_otra_entidad <= 11.86
|   |   |   |   |--- value: [8.05]
|   |   |   |--- pct_nacidos_otra_entidad >  11.86
|   |   |   |   |--- value: [8.27]
|   |   |--- dis_km >  2.58
|   |   |   |--- municipio_Zapopan <= 0.50
|   |   |   |   |--- value: [7.16]
|   |   |   |--- municipio_Zapopan >  0.50
|   |   |   |   |--- value: [7.92]
|   |--- graproes >  13.47
|   |   |--- restaurantes_cafes_1km <= 2.92
|   |   |   |--- value: [8.09]
|   |   |--- restaurantes_cafes_1km >  2.92
|   |   |   |--- graproes <= 14.72
|   |   |   |   |--- value: [8.67]
|   |   |   |--- graproes >  14.72
|   |   |   |   |--- value: [8.99]
|--- pro_ocup_c >  0.94
|   |--- pct_viv_3ymas_cuartos <= 81.43
|   |   |--- lst_c <= 44.82
|   |   |   |--- value: [6.44]
|   |   |--- lst_c >  44.82
|   |   |   |--- nox <= 60.82
|   |   |   |   |--- value: [7.04]
|   |   |   |--- nox >  60.82
|   |   |   |   |--- value: [7.63

**Transferencia:** se deja fuera un municipio entero y se mide el orden dentro de él. Es lo que haría falta para dar forma a un municipio sin tabla de valores.

In [27]:
candidatos = ["Lasso", "Aditivo (splines)", "PCR", "Mixto", "Red (32-16)", "Bosque"]
pd.concat([modelos.transferencia(cat[n][1], base, RASGOS).assign(modelo=n) for n in candidatos]
          ).pivot(index="municipio", columns="modelo", values="spearman")

modelo,Aditivo (splines),Bosque,Lasso,Mixto,PCR,Red (32-16)
municipio,,,,,,
Guadalajara,0.770,0.789,0.812,0.776,0.768,0.640
Ixtlahuacán de los Membrillos,-0.012,-0.045,0.022,-0.058,0.075,-0.041
Tonalá,0.448,0.417,0.492,0.414,0.468,0.435
Zapopan,0.645,0.619,0.669,0.637,0.654,0.583


## $\color{#dda}{\text{10. Rango conformal con un modelo interpretable}}$

El rango conformal no depende del modelo. Se compara el del estudio principal (bosque para el centro y para el ancho) contra Lasso para el centro con el ancho de un bosque, y Lasso para el centro con el ancho de una ridge sobre log|residuo|, que también se puede leer.

In [28]:
variantes = {
    "Bosque + ancho bosque": lambda: conformal.Normalizado(cat["Bosque"][1], RASGOS),
    "Lasso + ancho bosque": lambda: conformal.Normalizado(cat["Lasso"][1], RASGOS),
    "Lasso + ancho lineal": lambda: conformal.Normalizado(cat["Lasso"][1], RASGOS, escala="lineal"),
}
rangos = {n: conformal.evaluar(f, base, RASGOS, ALFA) for n, f in variantes.items()}
cob = pd.concat([conformal.resumen(r).assign(metodo=n) for n, r in rangos.items()])
cob.pivot(index="municipio", columns="metodo", values=["cobertura", "alto_entre_bajo_mediana"])

cobertura                       \
metodo                        Bosque + ancho bosque Lasso + ancho bosque   
municipio                                                                  
Guadalajara                                   0.786                0.740   
Ixtlahuacán de los Membrillos                 0.826                0.783   
Todos                                         0.796                0.767   
Tonalá                                        0.836                0.809   
Zapopan                                       0.788                0.781   

                                                   alto_entre_bajo_mediana  \
metodo                        Lasso + ancho lineal   Bosque + ancho bosque   
municipio                                                                    
Guadalajara                                  0.834                   1.488   
Ixtlahuacán de los Membrillos                0.783                  10.262   
Todos                                        0.778                   3.124   
Tonalá                                       0.750                   5.177   
Zapopan                                      0.715                   5.196   

                                                                         
metodo                        Lasso + ancho bosque Lasso + ancho lineal  
municipio                                                                
Guadalajara                                  1.604                1.767  
Ixtlahuacán de los Membrillos               12.900               12.134  
Todos                                        2.821                2.518  
Tonalá                                       4.485                3.514  
Zapopan                                      6.176                4.087

In [29]:
graficas.cobertura(cob, 1 - ALFA)

## $\color{#dda}{\text{11. Conclusiones}}$

**Lasso es el mejor modelo, y es interpretable.** Fuera de muestra por bloques tiene el menor error (RMSE en log 0.619) y el mejor orden (Spearman 0.738), por delante del bosque (0.622 y 0.720) y de la SVR (0.636 y 0.735). También transfiere mejor a un municipio que no vio (Spearman 0.81 en Guadalajara y 0.67 en Zapopan). Aquí ser interpretable no cuesta precisión.

**Lo que mueve el valor** coincide en la OLS por pasos, la Lasso y la logística:

| Rasgo | Efecto por una desviación estándar |
|---|---|
| Escolaridad promedio (`graproes`) | +10 a +13 %; multiplica por 2.3 los momios de estar arriba de la mediana del municipio |
| Ocupantes por cuarto (`pro_ocup_c`) | −12 a −13 % |
| Pendiente del terreno | −6 a −10 % |
| Hectáreas de parque a 1 km | +7 a +10 % |
| Viviendas de uso temporal | −5 a −7 % |
| Distancia a vías rápidas (`rad_km`) | −3 a −7 % |

**Por tema:**

- **Distribución:** el logaritmo baja el sesgo de 2.7 a −1.3. Queda una cola izquierda (los valores bajos del OCR de Zapopan y Tonalá) que se ve en el QQ de los residuos.
- **Regresión lineal:** la OLS completa tiene R² de 0.46, pero con 44 rasgos colineales (VIF altos) muchos intervalos cruzan el cero. Por pasos con BIC quedan 11 rasgos con signos estables, y en general su error es mejor que el de la OLS completa (0.634 contra 0.663).
- **ANOVA y ANCOVA:** el municipio solo explica 11 % de la varianza (η² = 0.11, F = 37.8). Con las covariables deja de ser significativo (p = 0.27): la diferencia entre municipios sale de cómo es cada zona y quién vive ahí, no del municipio en sí. Las pendientes paralelas se rechazan apenas (p = 0.044): el efecto de internet en la vivienda cambia un poco entre municipios.
- **PCA:** hacen falta 13 componentes para 80 % de la varianza. El primero es centralidad (cerca del centro, con servicios, antiguo contra lejos y hacinado), el segundo nivel socioeconómico, el tercero densidad. PCR queda a media tabla (0.643).
- **k-medias:** seis tipos de zona con medianas de $950 a $5,463/m². El tipo de 13 AGEB con $950 junta buena parte de las colas bajas del OCR. Como modelo es de los peores (Spearman 0.615): un tipo es demasiado grueso.
- **Logística:** AUC fuera de muestra de 0.80 para "¿vale más que la mediana de su municipio?".
- **Fuera del temario:** el aditivo con splines empata con el bosque (0.629) y deja leer la curva de cada rasgo. El árbol elegido (cuatro niveles) se lee como reglas (primero ocupantes por cuarto, después escolaridad), pero predice mal (0.689). El mixto casi no cambia nada frente a la OLS con cuatro municipios; su uso está en calibrar contra promedios oficiales.
- **Red neuronal (32 y 16 neuronas):** la peor en error (0.698). Con 892 AGEB no tiene datos para ganarle a un modelo lineal penalizado.

**Rango conformal:** con Lasso al centro y una ridge para el ancho, el rango es más angosto que el del bosque (alto entre bajo, mediana 2.5 contra 3.1; Zapopan 4.1 contra 5.2; Tonalá 3.5 contra 5.2). Pero cubre 77.8 % en vez de 79.6 %, y Zapopan baja a 71.5 %. Antes de cambiar el modelo de producción hay que cerrar esa diferencia, por ejemplo calibrando por municipio (conformal de Mondrian).